In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
rf_improved = joblib.load("salespulse_rf_model.pkl")
loaded_features = joblib.load("salespulse_features.pkl")

In [4]:

sales = pd.read_csv(
    "../data/raw/sales_train_evaluation.csv"
)

print(sales.shape)

(30490, 1947)


In [5]:
sales_cols = [
    col for col in sales.columns
    if col.startswith("d_")
]

print("Number of daily columns:", len(sales_cols))
print("First 5:", sales_cols[:5])
print("Last 5:", sales_cols[-5:])

Number of daily columns: 1941
First 5: ['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
Last 5: ['d_1937', 'd_1938', 'd_1939', 'd_1940', 'd_1941']


In [6]:
daily_totals = sales[sales_cols].sum(axis=0)

print(daily_totals.head())
print(daily_totals.tail())

d_1    32631
d_2    31749
d_3    23783
d_4    25412
d_5    19146
dtype: int64
d_1937    37096
d_1938    36963
d_1939    42552
d_1940    51518
d_1941    54338
dtype: int64


In [7]:
daily_sales = pd.DataFrame({
    "d": daily_totals.index,
    "total_sales": daily_totals.values
})

print(daily_sales.head())
print(daily_sales.shape)

     d  total_sales
0  d_1        32631
1  d_2        31749
2  d_3        23783
3  d_4        25412
4  d_5        19146
(1941, 2)


In [8]:
calendar = pd.read_csv("../data/raw/calendar.csv")

calendar["date"] = pd.to_datetime(calendar["date"])

print(calendar.head())
print(calendar.shape)

        date  wm_yr_wk    weekday  wday  month  year    d event_name_1  \
0 2011-01-29     11101   Saturday     1      1  2011  d_1          NaN   
1 2011-01-30     11101     Sunday     2      1  2011  d_2          NaN   
2 2011-01-31     11101     Monday     3      1  2011  d_3          NaN   
3 2011-02-01     11101    Tuesday     4      2  2011  d_4          NaN   
4 2011-02-02     11101  Wednesday     5      2  2011  d_5          NaN   

  event_type_1 event_name_2 event_type_2  snap_CA  snap_TX  snap_WI  
0          NaN          NaN          NaN        0        0        0  
1          NaN          NaN          NaN        0        0        0  
2          NaN          NaN          NaN        0        0        0  
3          NaN          NaN          NaN        1        1        0  
4          NaN          NaN          NaN        1        0        1  
(1969, 14)


In [9]:
daily_sales = daily_sales.merge(
    calendar,
    on="d",
    how="left"
)

print(daily_sales.head())
print(daily_sales.shape)

     d  total_sales       date  wm_yr_wk    weekday  wday  month  year  \
0  d_1        32631 2011-01-29     11101   Saturday     1      1  2011   
1  d_2        31749 2011-01-30     11101     Sunday     2      1  2011   
2  d_3        23783 2011-01-31     11101     Monday     3      1  2011   
3  d_4        25412 2011-02-01     11101    Tuesday     4      2  2011   
4  d_5        19146 2011-02-02     11101  Wednesday     5      2  2011   

  event_name_1 event_type_1 event_name_2 event_type_2  snap_CA  snap_TX  \
0          NaN          NaN          NaN          NaN        0        0   
1          NaN          NaN          NaN          NaN        0        0   
2          NaN          NaN          NaN          NaN        0        0   
3          NaN          NaN          NaN          NaN        1        1   
4          NaN          NaN          NaN          NaN        1        0   

   snap_WI  
0        0  
1        0  
2        0  
3        0  
4        1  
(1941, 15)


In [10]:
daily_sales["is_event_day"] = (
    daily_sales["event_name_1"].notna() |
    daily_sales["event_name_2"].notna()
).astype(int)

In [11]:
daily_sales["lag_1"] = daily_sales["total_sales"].shift(1)
daily_sales["lag_7"] = daily_sales["total_sales"].shift(7)
daily_sales["lag_14"] = daily_sales["total_sales"].shift(14)
daily_sales["lag_28"] = daily_sales["total_sales"].shift(28)

In [12]:
daily_sales["rolling_mean_7"] = (
    daily_sales["total_sales"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

daily_sales["rolling_mean_28"] = (
    daily_sales["total_sales"]
    .shift(1)
    .rolling(window=28)
    .mean()
)

In [13]:
daily_sales["year"] = daily_sales["date"].dt.year
daily_sales["month"] = daily_sales["date"].dt.month
daily_sales["day"] = daily_sales["date"].dt.day
daily_sales["day_of_week"] = daily_sales["date"].dt.dayofweek
daily_sales["week_of_year"] = daily_sales["date"].dt.isocalendar().week.astype(int)

In [14]:
prediction_features = [
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "is_event_day",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

print(daily_sales[prediction_features].tail())

      year  month  day  day_of_week  week_of_year  is_event_day    lag_1  \
1936  2016      5   18            2            20             0  38777.0   
1937  2016      5   19            3            20             0  37096.0   
1938  2016      5   20            4            20             0  36963.0   
1939  2016      5   21            5            20             0  42552.0   
1940  2016      5   22            6            20             0  51518.0   

        lag_7   lag_14   lag_28  rolling_mean_7  rolling_mean_28  
1936  42248.0  39601.0  35343.0    46027.428571     43533.821429  
1937  40503.0  40763.0  35033.0    45291.428571     43596.428571  
1938  44073.0  43805.0  40517.0    44785.714286     43665.357143  
1939  54308.0  54239.0  48962.0    44568.428571     43738.035714  
1940  59921.0  45609.0  49795.0    44169.857143     43829.321429  


In [15]:
prediction_data = daily_sales.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28"
    ]
).copy()

print("Prediction rows:", len(prediction_data))

Prediction rows: 1913


In [19]:
X = prediction_data[prediction_features]
y = prediction_data["total_sales"]

In [20]:
train_mask = prediction_data["date"] < "2015-01-01"

val_mask = (
    (prediction_data["date"] >= "2015-01-01") &
    (prediction_data["date"] < "2016-01-01")
)

test_mask = prediction_data["date"] >= "2016-01-01"

In [21]:
X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_val = X.loc[val_mask]
y_val = y.loc[val_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

In [22]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1405, 12)
Validation: (365, 12)
Test: (143, 12)


In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [24]:
lr = LinearRegression()

lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [25]:
lr_pred = lr.predict(X_val)

In [26]:
lr_mae = mean_absolute_error(y_val, lr_pred)

lr_rmse = np.sqrt(
    mean_squared_error(y_val, lr_pred)
)

lr_r2 = r2_score(y_val, lr_pred)

print("Linear Regression")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)
print("R²:", lr_r2)

Linear Regression
MAE: 2063.3552304956816
RMSE: 3134.1517434118678
R²: 0.7597552841436533


kNN Model

In [27]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(X_train)
x_val_scaled=scaler.transform(X_val)



In [50]:
x_test_scaled=scaler.transform(X_test)

In [29]:
knn = KNeighborsRegressor(
    n_neighbors=5
)

knn.fit(x_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [31]:
knn_pred = knn.predict(x_val_scaled)

In [32]:
knn_mae = mean_absolute_error(y_val, knn_pred)

knn_rmse = np.sqrt(
    mean_squared_error(y_val, knn_pred)
)

knn_r2 = r2_score(y_val, knn_pred)

print("KNN Regression")
print("MAE:", knn_mae)
print("RMSE:", knn_rmse)
print("R²:", knn_r2)

KNN Regression
MAE: 2105.8712328767124
RMSE: 3030.9670057454946
R²: 0.7753138913159143


Decision_Tree


In [33]:
from sklearn.tree import DecisionTreeRegressor

In [34]:
dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_l

In [35]:
dt_pred = dt.predict(X_val)

In [36]:
dt_mae = mean_absolute_error(y_val, dt_pred)

dt_rmse = np.sqrt(
    mean_squared_error(y_val, dt_pred)
)

dt_r2 = r2_score(y_val, dt_pred)

print("Decision Tree")
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)
print("R²:", dt_r2)

Decision Tree
MAE: 2809.621917808219
RMSE: 4669.169330951266
R²: 0.466796423636585


Random_Forest

In [37]:
from sklearn.ensemble import RandomForestRegressor

In [38]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [39]:
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [40]:
rf_pred = rf.predict(X_val)

In [41]:
rf_mae = mean_absolute_error(y_val, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_val, rf_pred)
)

rf_r2 = r2_score(y_val, rf_pred)

print("Random Forest")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R²:", rf_r2)

Random Forest
MAE: 1978.4252602739725
RMSE: 3143.3993228498034
R²: 0.758335467933773


GradentBoostingRegressor

In [42]:
from sklearn.ensemble import GradientBoostingRegressor

In [43]:
gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [44]:
gb.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [45]:
gb_pred = gb.predict(X_val)

In [46]:
gb_mae = mean_absolute_error(y_val, gb_pred)

gb_rmse = np.sqrt(
    mean_squared_error(y_val, gb_pred)
)

gb_r2 = r2_score(y_val, gb_pred)

print("Gradient Boosting")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R²:", gb_r2)

Gradient Boosting
MAE: 1863.4482631774358
RMSE: 2806.945971890562
R²: 0.8072999124106077


In [47]:
results = [
    {
        "Model": "Linear Regression",
        "MAE": lr_mae,
        "RMSE": lr_rmse,
        "R2": lr_r2
    },
    {
        "Model": "KNN",
        "MAE": knn_mae,
        "RMSE": knn_rmse,
        "R2": knn_r2
    },
    {
        "Model": "Decision Tree",
        "MAE": dt_mae,
        "RMSE": dt_rmse,
        "R2": dt_r2
    },
    {
        "Model": "Random Forest",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "R2": rf_r2
    },
    {
        "Model": "Gradient Boosting",
        "MAE": gb_mae,
        "RMSE": gb_rmse,
        "R2": gb_r2
    }
]

results_df = pd.DataFrame(results)

print(results_df)

               Model          MAE         RMSE        R2
0  Linear Regression  2063.355230  3134.151743  0.759755
1                KNN  2105.871233  3030.967006  0.775314
2      Decision Tree  2809.621918  4669.169331  0.466796
3      Random Forest  1978.425260  3143.399323  0.758335
4  Gradient Boosting  1863.448263  2806.945972  0.807300


In [48]:
results_df.sort_values("MAE")

,Model,MAE,RMSE,R2
4,Gradient Boosting,1863.448263,2806.945972,0.807300
3,Random Forest,1978.425260,3143.399323,0.758335
0,Linear Regression,2063.355230,3134.151743,0.759755
1,KNN,2105.871233,3030.967006,0.775314
2,Decision Tree,2809.621918,4669.169331,0.466796


In [51]:
lr_test_pred = lr.predict(X_test)

knn_test_pred = knn.predict(x_test_scaled)

dt_test_pred = dt.predict(X_test)

rf_test_pred = rf.predict(X_test)

gb_test_pred = gb.predict(X_test)

In [52]:
test_results = []

models_predictions = {
    "Linear Regression": lr_test_pred,
    "KNN": knn_test_pred,
    "Decision Tree": dt_test_pred,
    "Random Forest": rf_test_pred,
    "Gradient Boosting": gb_test_pred
}

for model_name, predictions in models_predictions.items():

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    r2 = r2_score(y_test, predictions)

    test_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

test_results_df = pd.DataFrame(test_results)

print(test_results_df)

               Model          MAE         RMSE        R2
0  Linear Regression  2250.102672  3035.748428  0.805433
1                KNN  3964.482517  4864.606859  0.500389
2      Decision Tree  4261.209790  5482.494442  0.365410
3      Random Forest  2665.684476  3498.349726  0.741618
4  Gradient Boosting  2467.506393  3322.776623  0.766902


In [53]:
comparison_df = results_df.merge(
    test_results_df,
    on="Model",
    suffixes=("_Validation", "_Test")
)

print(comparison_df)

               Model  MAE_Validation  RMSE_Validation  R2_Validation  \
0  Linear Regression     2063.355230      3134.151743       0.759755   
1                KNN     2105.871233      3030.967006       0.775314   
2      Decision Tree     2809.621918      4669.169331       0.466796   
3      Random Forest     1978.425260      3143.399323       0.758335   
4  Gradient Boosting     1863.448263      2806.945972       0.807300   

      MAE_Test    RMSE_Test   R2_Test  
0  2250.102672  3035.748428  0.805433  
1  3964.482517  4864.606859  0.500389  
2  4261.209790  5482.494442  0.365410  
3  2665.684476  3498.349726  0.741618  
4  2467.506393  3322.776623  0.766902  


In [54]:
comparison_df["MAE_Change"] = (
    comparison_df["MAE_Test"] -
    comparison_df["MAE_Validation"]
)

print(
    comparison_df[
        ["Model", "MAE_Validation", "MAE_Test", "MAE_Change"]
    ]
)

               Model  MAE_Validation     MAE_Test   MAE_Change
0  Linear Regression     2063.355230  2250.102672   186.747442
1                KNN     2105.871233  3964.482517  1858.611285
2      Decision Tree     2809.621918  4261.209790  1451.587872
3      Random Forest     1978.425260  2665.684476   687.259215
4  Gradient Boosting     1863.448263  2467.506393   604.058129


combine training and validation data

In [55]:
X_train_final = pd.concat(
    [X_train, X_val]
)

y_train_final = pd.concat(
    [y_train, y_val]
)

print("Final training shape:", X_train_final.shape)
print("Final target shape:", y_train_final.shape)

Final training shape: (1770, 12)
Final target shape: (1770,)


Final Model = Linear_Regression

In [56]:
final_model = LinearRegression()

final_model.fit(
    X_train_final,
    y_train_final
)

print("Final model trained successfully!")

Final model trained successfully!


In [57]:
final_test_pred = final_model.predict(X_test)

In [58]:
final_mae = mean_absolute_error(
    y_test,
    final_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(y_test, final_test_pred)
)

final_r2 = r2_score(
    y_test,
    final_test_pred
)

print("Final Linear Regression Results")
print("MAE:", final_mae)
print("RMSE:", final_rmse)
print("R²:", final_r2)

Final Linear Regression Results
MAE: 2241.7775391023406
RMSE: 2979.5951029310077
R²: 0.8125648021994085


In [59]:
import joblib

joblib.dump(
    final_model,
    "salespulse_final_model.pkl"
)

print("Final model saved successfully!")

Final model saved successfully!


In [60]:
joblib.dump(
   prediction_features,
    "salespulse_features.pkl"
)

print("Feature list saved successfully!")

Feature list saved successfully!
